# Dijet Embedding versus PYTHIA

Compare unit-normalized dijet $\eta_{CM}^{dijet}$ distributions and unnormalized Forward/Backward ratios between Embedding and PYTHIA. Each comparison overlays the two samples and shows PYTHIA / Embedding in the lower panel for every interval in `TEST_DIJET_PTAVE_BINS`. Reco, matched Ref, and Gen are run from separate cells.

## Environment and imports

Start Jupyter from the repository root with `py-env/bin/python -m jupyter notebook`.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os
import sys
from IPython.display import display

PROJECT_ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / 'CMakeLists.txt').is_file()
     and (candidate / 'hist_analysis').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Cannot locate the jetAnalysis repository. Start Jupyter from its root.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hist_analysis.python.notebook_setup import load_root
ROOT = load_root(batch=True)

from hist_analysis.config.files import BASE_DIR
from hist_analysis.config.histograms import (
    DIJET_DELTA_PHI_SELECTION_LABEL, TEST_DIJET_PTAVE_BINS,
)
from hist_analysis.python.dijet_closures import (
    DijetClosureCurve, build_dijet_gen_comparisons,
)
from hist_analysis.python.histogram_io import (
    resolve_combined_file, resolve_direction_file,
)
from hist_analysis.python.plotting import draw_closure

In [ ]:
ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetPalette(ROOT.kBird)
ROOT.TH1.AddDirectory(False)

## Configuration

`ETA_CUT` selects the stored jet-acceptance index. Full distributions are normalized independently to unit bin-content sum. Their common y-axis range is derived once from all Reco, Ref, and Gen projections in all test pTave intervals. Forward and backward yields are never normalized before forming F/B, and F/B always uses ROOT independent-error propagation. F/B overlays use the single configured `FB_RANGE`; the later PYTHIA / Embedding ratios also use independent errors.

In [ ]:
DIRECTION = 'combined'       # combined, Pbgoing, or pgoing
FILE_STEM = 'jetId'
ETA_CUTS = (1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.5)
ETA_CUT = 1.9
REBIN_ETA = 2
NORMALIZATION = 'integral'
ETA_RATIO_RANGE = (0.85, 1.15)
FB_RANGE = (0.75, 1.30)
FB_COMPARISON_RATIO_RANGE = (0.85, 1.15)
DRAW_GRID = True
SAVE_PNG = False
OUTPUT_DIR = Path(os.environ.get(
    'DIJET_EMBEDDING_VS_PYTHIA_OUTPUT_DIR',
    PROJECT_ROOT / 'hist_analysis' / 'output' / 'dijet_embedding_vs_pythia',
))

try:
    ETA_CUT_INDEX = ETA_CUTS.index(ETA_CUT)
except ValueError as error:
    raise ValueError(f'ETA_CUT must be one of {ETA_CUTS}, got {ETA_CUT!r}') from error
if DIRECTION not in {'combined', 'Pbgoing', 'pgoing'}:
    raise ValueError(f'Unsupported DIRECTION={DIRECTION!r}')
if isinstance(REBIN_ETA, bool) or not isinstance(REBIN_ETA, int) or REBIN_ETA < 1:
    raise ValueError('REBIN_ETA must be a positive integer')

In [ ]:
def mc_file(generator):
    if DIRECTION == 'combined':
        return resolve_combined_file(BASE_DIR, generator, FILE_STEM)
    return resolve_direction_file(BASE_DIR, generator, DIRECTION, FILE_STEM)

INPUT_FILES = {
    'Embedding': mc_file('embedding'),
    'PYTHIA': mc_file('pythia'),
}
for sample, filename in INPUT_FILES.items():
    if not filename.exists():
        raise FileNotFoundError(f'Missing {sample} ROOT file: {filename}')
INPUT_FILES

## Comparison helper

The stored full, Forward, and Backward TH2 histograms are projected over the same half-open $p_T^{ave}$ interval. F/B is constructed inside `build_dijet_gen_comparisons` with `TH1::Divide(..., '')`; no binomial option is exposed.

In [ ]:
LEVELS = {
    'Reco': DijetClosureCurve(
        'Reco', 'hRecoDijetPtEtaCM_{eta_cut_index}',
        'hRecoDijetPtEtaForward_{eta_cut_index}',
        'hRecoDijetPtEtaBackward_{eta_cut_index}',
    ),
    'Ref': DijetClosureCurve(
        'Ref', 'hRefDijetPtEtaCM_{eta_cut_index}',
        'hRefDijetPtEtaForward_{eta_cut_index}',
        'hRefDijetPtEtaBackward_{eta_cut_index}',
    ),
    'Gen': DijetClosureCurve(
        'Gen', 'hGenDijetPtEtaCM_{eta_cut_index}',
        'hGenDijetPtEtaForward_{eta_cut_index}',
        'hGenDijetPtEtaBackward_{eta_cut_index}',
    ),
}

DIRECTION_LABELS = {
    'combined': 'Combined', 'Pbgoing': 'Pb-going', 'pgoing': 'p-going',
}
STYLE_INDICES = {'Embedding': 0, 'PYTHIA': 1}

# Prepare every projection before drawing so every eta overlay can use one
# global y-axis range. The cached F/B histograms remain constructed from
# unnormalized Forward and Backward yields with independent ROOT errors.
PROJECTION_CACHE = {}
for level, curve in LEVELS.items():
    for ptave_range in TEST_DIJET_PTAVE_BINS:
        cache_entry = {'eta_shapes': {}, 'forward_backward': {}, 'keys': {}}
        for sample, filename in INPUT_FILES.items():
            sample_shapes, sample_fb, keys = build_dijet_gen_comparisons(
                filename, (curve,), eta_cut_index=ETA_CUT_INDEX,
                ptave_range=ptave_range, nominal=level,
                rebin_eta=REBIN_ETA, normalization=NORMALIZATION,
                ratio_option='',
            )
            cache_entry['eta_shapes'][sample] = sample_shapes[level]
            cache_entry['forward_backward'][sample] = sample_fb[level]
            cache_entry['keys'][sample] = keys[level]
        PROJECTION_CACHE[(level, tuple(ptave_range))] = cache_entry

eta_upper = max(
    histogram.GetBinContent(bin_index) + histogram.GetBinError(bin_index)
    for entry in PROJECTION_CACHE.values()
    for histogram in entry['eta_shapes'].values()
    for bin_index in range(1, histogram.GetNbinsX() + 1)
)
COMMON_ETA_Y_RANGE = (0.0, 1.15 * eta_upper)
print('Common eta-distribution y range:', COMMON_ETA_Y_RANGE)

def compare_level(level):
    if level not in LEVELS:
        raise KeyError(f'Unknown level {level!r}; choose from {tuple(LEVELS)}')
    level_tag = level.lower()
    results = {}
    for low, high in TEST_DIJET_PTAVE_BINS:
        ptave_range = (low, high)
        ptave_tag = f'{low:g}_{high:g}'.replace('.', 'p')
        cached = PROJECTION_CACHE[(level, ptave_range)]
        eta_shapes = cached['eta_shapes']
        fb_ratios = cached['forward_backward']
        selected_keys = cached['keys']

        annotations = (
            'pPb 8.16 TeV', f'{level} dijets', DIRECTION_LABELS[DIRECTION],
            f'{low:g} < p_{{T}}^{{ave}} < {high:g} GeV',
            f'|#eta_{{CM}}^{{jet}}| < {ETA_CUT:g}',
            'p_{T}^{Lead} > 50 GeV', 'p_{T}^{SubLead} > 40 GeV',
            DIJET_DELTA_PHI_SELECTION_LABEL,
        )
        eta_tag = int(round(10.0 * ETA_CUT))
        base_tag = f'{level_tag}_{DIRECTION}_etaCM_{eta_tag}_ptave_{ptave_tag}'
        eta_canvas, eta_ratio = draw_closure(
            eta_shapes, 'Embedding', title='', x_title='#eta_{CM}^{dijet}',
            y_title='1/N dN/d#eta_{CM}^{dijet}', ratio_range=ETA_RATIO_RANGE,
            x_range=(-ETA_CUT - 0.1, ETA_CUT + 0.1), y_range=COMMON_ETA_Y_RANGE,
            annotations=annotations, grid=DRAW_GRID, draw_nominal_ratio=False,
            ratio_option='', style_indices=STYLE_INDICES,
            output=OUTPUT_DIR / f'{base_tag}_distribution.pdf',
            save_png=SAVE_PNG, canvas_name=f'{base_tag}_distribution',
        )
        fb_canvas, fb_ratio = draw_closure(
            fb_ratios, 'Embedding', title='', x_title='|#eta_{CM}^{dijet}|',
            y_title='Forward / Backward', ratio_range=FB_COMPARISON_RATIO_RANGE,
            x_range=(0.0, ETA_CUT + 0.1), y_range=FB_RANGE,
            annotations=annotations, grid=DRAW_GRID, draw_nominal_ratio=False,
            ratio_option='', style_indices=STYLE_INDICES,
            output=OUTPUT_DIR / f'{base_tag}_forward_backward.pdf',
            save_png=SAVE_PNG, canvas_name=f'{base_tag}_forward_backward',
        )
        results[ptave_range] = {
            'eta_shapes': eta_shapes, 'pythia_over_embedding': eta_ratio['PYTHIA'],
            'forward_backward': fb_ratios,
            'fb_pythia_over_embedding': fb_ratio['PYTHIA'],
            'eta_canvas': eta_canvas, 'forward_backward_canvas': fb_canvas,
            'keys': selected_keys,
        }
        print(level, ptave_range, selected_keys)
        display(eta_canvas)
        display(fb_canvas)
    return results

## Reco comparison

In [ ]:
reco_results = compare_level('Reco')

## Ref comparison

In [ ]:
ref_results = compare_level('Ref')

## Gen comparison

In [ ]:
gen_results = compare_level('Gen')